# LNL-Ti: XGBoost-Light CutMix two-stage training

This notebook has two deliberately separate stages:

1. `development`: keep one track-safe validation split fixed and repeat the
   same configuration with training seeds `42`, `123`, and `3407`.
2. `full_refit`: start from scratch on all 39,209 training images, use the
   median best epoch from the three development runs, then test once.

Set `TRAINING_SEED` before each fresh development runtime. The third completed
seed automatically creates the locked configuration used by `full_refit`.

The XGBoost-Light schedule is CutMix regularization, learning-rate shrinkage,
then clean refinement. Mild downsampling/blur is used only in the CutMix phase.
ROI augmentation, hardness weighting, EMA, MoEx, RandAugment, and inference
ensembles remain disabled to keep the experiment focused and reproducible.

In [ ]:
from pathlib import Path
import os
import subprocess

# Keep this as development first; use full_refit only in a fresh runtime.
RUN_STAGE = os.environ.get("RUN_STAGE", "development").strip().lower()
TRAINING_SEED = int(os.environ.get("TRAINING_SEED", "42"))

# Pipeline 1/7 - Locate the repository without changing source notebooks.
REPOSITORY_URL = "https://github.com/Omid-Nejati/Locality-iN-Locality.git"
repository_root = next(
    (
        path
        for path in (
            Path.cwd(),
            Path.cwd() / "Locality-iN-Locality-main",
            Path.cwd() / "Locality-iN-Locality",
        )
        if (path / "LNL_MoEx.py").exists()
    ),
    None,
)

if repository_root is None:
    repository_root = Path.cwd() / "Locality-iN-Locality"
    subprocess.run(["git", "clone", REPOSITORY_URL, str(repository_root)], check=True)

os.chdir(repository_root)
print("Run stage:", RUN_STAGE)
print("Working directory:", Path.cwd())

In [ ]:
%pip install -q timm==0.9.16 einops scikit-learn==1.6.1 tqdm

In [ ]:
import csv
import json
import math
import random
import shutil
import zipfile
from dataclasses import dataclass
from urllib.request import urlretrieve

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

DEVELOPMENT_SEEDS = (42, 123, 3407)
SPLIT_SEED = 42
FULL_REFIT_SEED = 42
if RUN_STAGE not in {"development", "full_refit"}:
    raise ValueError("RUN_STAGE must be 'development' or 'full_refit'.")
if RUN_STAGE == "development" and TRAINING_SEED not in DEVELOPMENT_SEEDS:
    raise ValueError(f"TRAINING_SEED must be one of {DEVELOPMENT_SEEDS}.")
SEED = TRAINING_SEED if RUN_STAGE == "development" else FULL_REFIT_SEED


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Stage: {RUN_STAGE} | training seed: {SEED} | split seed: {SPLIT_SEED}")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("Device:", device)

## Data

In [ ]:
# Pipeline 2/7 - Download, extract, and arrange the official GTSRB split.
DATA_DIR = Path("data")
ARCHIVE_BASE_URL = (
    "https://sid.erda.dk/public/archives/"
    "daaeac0d7ce1152aea9b61d9f1e19370"
)
ARCHIVES = (
    "GTSRB_Final_Training_Images.zip",
    "GTSRB_Final_Test_Images.zip",
    "GTSRB_Final_Test_GT.zip",
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
required_paths = (
    DATA_DIR / "GTSRB" / "Final_Training" / "Images",
    DATA_DIR / "GTSRB" / "Final_Test" / "Images",
    DATA_DIR / "GT-final_test.csv",
)

if not all(path.exists() for path in required_paths):
    for filename in ARCHIVES:
        destination = DATA_DIR / filename
        if not destination.exists():
            urlretrieve(f"{ARCHIVE_BASE_URL}/{filename}", destination)
        with zipfile.ZipFile(destination) as archive:
            archive.extractall(DATA_DIR)

test_images_dir = DATA_DIR / "GTSRB" / "Final_Test" / "Images"
test_root = DATA_DIR / "GTSRB" / "test"
test_root.mkdir(parents=True, exist_ok=True)

with (DATA_DIR / "GT-final_test.csv").open(newline="") as csv_file:
    for row in csv.DictReader(csv_file, delimiter=";"):
        class_dir = test_root / f"{int(row['ClassId']):04d}"
        destination = class_dir / row["Filename"]
        if not destination.exists():
            class_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(test_images_dir / row["Filename"], destination)

TRAIN_ROOT = DATA_DIR / "GTSRB" / "Final_Training" / "Images"
TEST_ROOT = DATA_DIR / "GTSRB" / "test"

print("Dataset root:", (DATA_DIR / "GTSRB").resolve())

In [ ]:
# Pipeline 3/7 - Mild detail loss is explicit, mutually exclusive, and phase-limited.
REGULARIZED_AUGMENTATION = {
    "affine": {
        "degrees": 5,
        "translate": [0.02, 0.02],
        "scale": [0.97, 1.05],
        "probability": 0.50,
    },
    "detail_degradation": {
        "probability": 0.20,
        "downsample_share": 0.75,
        "downsample_scale": [0.70, 0.90],
        "blur_kernel": 3,
        "blur_sigma": [0.10, 0.80],
    },
    "color": {
        "brightness": 0.15,
        "contrast": 0.15,
        "saturation": 0.10,
        "hue": 0.01,
        "probability": 0.40,
    },
}
COOLDOWN_AUGMENTATION = {
    "affine": {
        "degrees": 3,
        "translate": [0.01, 0.01],
        "scale": [0.98, 1.03],
        "probability": 0.35,
    },
    "color": {
        "brightness": 0.08,
        "contrast": 0.08,
        "saturation": 0.05,
        "hue": 0.005,
        "probability": 0.25,
    },
}


class ResizeWithPadding:
    def __init__(self, size=224, fill=128):
        self.size = size
        self.fill = fill

    def __call__(self, image):
        width, height = image.size
        scale = self.size / max(width, height)
        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))
        image = TF.resize(
            image,
            [new_height, new_width],
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        )
        horizontal = self.size - new_width
        vertical = self.size - new_height
        return TF.pad(
            image,
            [horizontal // 2, vertical // 2,
             horizontal - horizontal // 2, vertical - vertical // 2],
            fill=self.fill,
        )


class RandomDetailDegradation:
    def __init__(self, config):
        self.config = config

    def __call__(self, image):
        if random.random() >= self.config["probability"]:
            return image
        if random.random() >= self.config["downsample_share"]:
            sigma = random.uniform(*self.config["blur_sigma"])
            return TF.gaussian_blur(
                image,
                kernel_size=self.config["blur_kernel"],
                sigma=sigma,
            )

        width, height = image.size
        ratio = random.uniform(*self.config["downsample_scale"])
        reduced = TF.resize(
            image,
            [max(1, round(height * ratio)), max(1, round(width * ratio))],
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        )
        return TF.resize(
            reduced,
            [height, width],
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        )


normalize = transforms.Normalize(
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
)
eval_transform = transforms.Compose(
    [ResizeWithPadding(224), transforms.ToTensor(), normalize]
)


def build_train_transform(profile):
    affine = profile["affine"]
    color = profile["color"]
    operations = [
        ResizeWithPadding(224),
        transforms.RandomApply(
            [
                transforms.RandomAffine(
                    degrees=affine["degrees"],
                    translate=affine["translate"],
                    scale=affine["scale"],
                    interpolation=transforms.InterpolationMode.BILINEAR,
                    fill=128,
                )
            ],
            p=affine["probability"],
        ),
    ]
    if "detail_degradation" in profile:
        operations.append(RandomDetailDegradation(profile["detail_degradation"]))
    operations.extend(
        [
            transforms.RandomApply(
                [
                    transforms.ColorJitter(
                        brightness=color["brightness"],
                        contrast=color["contrast"],
                        saturation=color["saturation"],
                        hue=color["hue"],
                    )
                ],
                p=color["probability"],
            ),
            transforms.ToTensor(),
            normalize,
        ]
    )
    return transforms.Compose(operations)


regularized_train_transform = build_train_transform(REGULARIZED_AUGMENTATION)
cooldown_train_transform = build_train_transform(COOLDOWN_AUGMENTATION)

In [ ]:
# Pipeline 4/7 - Development splits by track; full refit uses every image.
TARGET_VALIDATION_SIZE = 4000
MICRO_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 0

index_dataset = torchvision.datasets.ImageFolder(TRAIN_ROOT)
indices = np.arange(len(index_dataset))
targets = np.asarray(index_dataset.targets)


def track_id(sample_path):
    path = Path(sample_path)
    parts = path.stem.split("_")
    if len(parts) != 2 or not all(part.isdigit() for part in parts):
        raise ValueError(
            f"Unexpected GTSRB filename {path.name!r}; "
            "expected <TrackID>_<FrameID>.ppm."
        )
    return f"{path.parent.name}:{parts[0]}"


if RUN_STAGE == "development":
    groups = np.asarray([track_id(path) for path, _ in index_dataset.samples])
    min_tracks_per_class = min(
        np.unique(groups[targets == class_id]).size
        for class_id in range(len(index_dataset.classes))
    )
    n_splits = min(
        max(2, round(len(index_dataset) / TARGET_VALIDATION_SIZE)),
        min_tracks_per_class,
    )
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SPLIT_SEED,
    )
    training_indices, validation_indices = min(
        splitter.split(indices, targets, groups),
        key=lambda split: abs(len(split[1]) - TARGET_VALIDATION_SIZE),
    )
    if set(groups[training_indices]) & set(groups[validation_indices]):
        raise RuntimeError("Track leakage detected between train and validation.")
else:
    n_splits = None
    training_indices = indices
    validation_indices = None

training_dataset = torchvision.datasets.ImageFolder(
    TRAIN_ROOT,
    transform=regularized_train_transform,
)
validation_dataset = (
    torchvision.datasets.ImageFolder(TRAIN_ROOT, transform=eval_transform)
    if RUN_STAGE == "development"
    else None
)

loader_generator = torch.Generator().manual_seed(SEED)
loader_options = {
    "num_workers": NUM_WORKERS,
    "pin_memory": device.type == "cuda",
}
training_loader = DataLoader(
    Subset(training_dataset, training_indices),
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    **loader_options,
)
validation_loader = (
    DataLoader(
        Subset(validation_dataset, validation_indices),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        **loader_options,
    )
    if RUN_STAGE == "development"
    else None
)

if RUN_STAGE == "development":
    print(f"Track-safe split: {n_splits} folds")
    print(
        f"Train: {len(training_indices):,} | "
        f"Validation: {len(validation_indices):,} | Test: reserved"
    )
else:
    print(f"Full refit train: {len(training_indices):,} | Validation: disabled")

## Train LNL-Ti with XGBoost-Light CutMix

In [ ]:
from LNL_MoEx import LNL_MoEx_Ti

NUM_CLASSES = 43
DEVELOPMENT_MAX_EPOCHS = 60
MIN_DEVELOPMENT_EPOCHS = 45
EARLY_STOP_PATIENCE = 12
TARGET_TOP1 = 99.5

CUTMIX_STOP_EPOCH = 30
CLEAN_START_EPOCH = 40
CUTMIX_PROBABILITY = 0.25
CUTMIX_ALPHA = 1.0
REGULARIZED_LABEL_SMOOTHING = 0.01
CLEAN_LABEL_SMOOTHING = 0.0

WARMUP_EPOCHS = 5
ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-4
COOLDOWN_LR_FACTOR = 0.10
MIN_LR_FACTOR = 0.01
WEIGHT_DECAY = 0.05
GRAD_CLIP_NORM = 1.0
DROP_PATH_RATE = 0.10
FORMAT_VERSION = 11
EXPERIMENT_NAME = "LNL_Ti_xgb_cutmix_v11"


def locked_config():
    return {
        "development_seeds": list(DEVELOPMENT_SEEDS),
        "split_seed": SPLIT_SEED,
        "full_refit_seed": FULL_REFIT_SEED,
        "cutmix_stop_epoch": CUTMIX_STOP_EPOCH,
        "clean_start_epoch": CLEAN_START_EPOCH,
        "cutmix_probability": CUTMIX_PROBABILITY,
        "cutmix_alpha": CUTMIX_ALPHA,
        "regularized_label_smoothing": REGULARIZED_LABEL_SMOOTHING,
        "clean_label_smoothing": CLEAN_LABEL_SMOOTHING,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": "linear_warmup_then_cosine",
        "warmup_epochs": WARMUP_EPOCHS,
        "cooldown_lr_factor": COOLDOWN_LR_FACTOR,
        "minimum_lr_factor": MIN_LR_FACTOR,
        "accumulation_steps": ACCUMULATION_STEPS,
        "gradient_clip_norm": GRAD_CLIP_NORM,
        "drop_path_rate": DROP_PATH_RATE,
        "roi_augmentation_probability": 0.0,
        "hardness_weighting": False,
        "evaluation": "single_view",
        "regularized_augmentation": REGULARIZED_AUGMENTATION,
        "cooldown_augmentation": COOLDOWN_AUGMENTATION,
    }


default_checkpoint_dir = (
    Path("/content/drive/MyDrive/LNL_MoEx_checkpoints")
    if Path("/content/drive/MyDrive").exists()
    else Path("checkpoints")
)
CHECKPOINT_DIR = Path(os.environ.get("CHECKPOINT_DIR", str(default_checkpoint_dir)))
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def development_result_path(seed):
    return CHECKPOINT_DIR / f"{EXPERIMENT_NAME}_seed{seed}_development.json"


DEVELOPMENT_RESULT_PATHS = {
    seed: development_result_path(seed) for seed in DEVELOPMENT_SEEDS
}
DEVELOPMENT_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / f"{EXPERIMENT_NAME}_seed{SEED}_development.pth"
)
FULL_REFIT_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / f"{EXPERIMENT_NAME}_seed{FULL_REFIT_SEED}_full_refit.pth"
)
LOCKED_CONFIG_PATH = CHECKPOINT_DIR / f"{EXPERIMENT_NAME}_multiseed_locked.json"

if RUN_STAGE == "development":
    NUM_EPOCHS = DEVELOPMENT_MAX_EPOCHS
    CHECKPOINT_PATH = DEVELOPMENT_CHECKPOINT_PATH
else:
    if not LOCKED_CONFIG_PATH.is_file():
        raise FileNotFoundError(
            "Complete all three development seeds first; lock is missing: "
            f"{LOCKED_CONFIG_PATH}"
        )
    locked = json.loads(LOCKED_CONFIG_PATH.read_text(encoding="utf-8"))
    if locked["config"] != locked_config():
        raise RuntimeError("Current hyperparameters differ from the locked runs.")
    NUM_EPOCHS = int(locked["full_refit_epochs"])
    if NUM_EPOCHS < 1:
        raise RuntimeError("Locked full_refit_epochs must be positive.")
    CHECKPOINT_PATH = FULL_REFIT_CHECKPOINT_PATH

RESUME_TRAINING = os.environ.get("RESUME_TRAINING", "1") == "1"


def build_model():
    return LNL_MoEx_Ti(
        pretrained=False,
        num_classes=NUM_CLASSES,
        drop_path_rate=DROP_PATH_RATE,
    ).to(device)


model = build_model()
decay_parameters = []
no_decay_parameters = []
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    if parameter.ndim <= 1 or name.endswith(".bias"):
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)

optimizer = optim.AdamW(
    [
        {"params": decay_parameters, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)
regularized_criterion = nn.CrossEntropyLoss(
    label_smoothing=REGULARIZED_LABEL_SMOOTHING
)
clean_criterion = nn.CrossEntropyLoss(
    label_smoothing=CLEAN_LABEL_SMOOTHING
)


def learning_rate_factor(epoch):
    if epoch < WARMUP_EPOCHS:
        factor = (epoch + 1) / WARMUP_EPOCHS
    else:
        progress = (epoch - WARMUP_EPOCHS) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        factor = MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * cosine
    return factor * (COOLDOWN_LR_FACTOR if epoch >= CUTMIX_STOP_EPOCH else 1.0)


scheduler = optim.lr_scheduler.LambdaLR(optimizer, learning_rate_factor)
scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")
print("Optimizer: AdamW | scheduler: 5-epoch warmup + cosine decay")
print("Epochs:", NUM_EPOCHS)
print("Checkpoint:", CHECKPOINT_PATH)

In [ ]:
@dataclass
class TrainState:
    epoch: int = 0
    best_epoch: int = -1
    best_correct: int = -1
    best_top1: float = -1.0
    best_weights: dict = None
    stale_epochs: int = 0
    cooldown_started: bool = False
    clean_phase_started: bool = False
    official_test_result: dict = None


@torch.inference_mode()
def evaluate_top1(module, loader):
    module.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=device.type == "cuda"):
            predictions = module(images).argmax(dim=1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)
    return correct, total, 100.0 * correct / total

In [ ]:
def cpu_state_dict(module):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in module.state_dict().items()
    }


def load_torch_file(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


# Pipeline 5/7 - Checkpoints are isolated by stage and training seed.
def save_checkpoint(state):
    payload = {
        "format_version": FORMAT_VERSION,
        "run_stage": RUN_STAGE,
        "run_seed": SEED,
        "epoch": state.epoch,
        "best_epoch": state.best_epoch,
        "best_validation_correct": state.best_correct,
        "best_validation_top1": state.best_top1,
        "best_model_state_dict": state.best_weights,
        "latest_model_state_dict": cpu_state_dict(model),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "stale_epochs": state.stale_epochs,
        "cooldown_started": state.cooldown_started,
        "clean_phase_started": state.clean_phase_started,
        "official_test_result": state.official_test_result,
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        ),
        "loader_rng_state": loader_generator.get_state(),
        "config": locked_config(),
    }
    temporary_path = CHECKPOINT_PATH.with_suffix(CHECKPOINT_PATH.suffix + ".tmp")
    torch.save(payload, temporary_path)
    temporary_path.replace(CHECKPOINT_PATH)


def load_checkpoint():
    if not RESUME_TRAINING or not CHECKPOINT_PATH.exists():
        return TrainState()

    checkpoint = load_torch_file(CHECKPOINT_PATH)
    if checkpoint.get("format_version") != FORMAT_VERSION:
        raise ValueError("Checkpoint format mismatch; use the v11 paths.")
    if checkpoint.get("run_stage") != RUN_STAGE:
        raise ValueError("Checkpoint stage does not match RUN_STAGE.")
    if checkpoint.get("run_seed") != SEED:
        raise ValueError("Checkpoint seed does not match TRAINING_SEED.")
    if checkpoint.get("config") != locked_config():
        raise ValueError("Checkpoint hyperparameters differ from this notebook.")

    model.load_state_dict(checkpoint["latest_model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    random.setstate(checkpoint["python_rng_state"])
    np.random.set_state(checkpoint["numpy_rng_state"])
    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    loader_generator.set_state(checkpoint["loader_rng_state"].cpu())
    cuda_states = checkpoint["cuda_rng_states"]
    if torch.cuda.is_available() and cuda_states is not None:
        torch.cuda.set_rng_state_all([value.cpu() for value in cuda_states])

    print(f"Resuming {RUN_STAGE} seed {SEED} from epoch {checkpoint['epoch'] + 1}.")
    return TrainState(
        epoch=checkpoint["epoch"],
        best_epoch=checkpoint.get("best_epoch", -1),
        best_correct=checkpoint["best_validation_correct"],
        best_top1=checkpoint["best_validation_top1"],
        best_weights=checkpoint["best_model_state_dict"],
        stale_epochs=checkpoint["stale_epochs"],
        cooldown_started=bool(checkpoint.get("cooldown_started", False)),
        clean_phase_started=bool(checkpoint.get("clean_phase_started", False)),
        official_test_result=checkpoint.get("official_test_result"),
    )

In [ ]:
# XGBoost-Light controller: regularize, shrink updates, then clean refinement.
def training_phase(epoch):
    if epoch < CUTMIX_STOP_EPOCH:
        return "cutmix"
    if epoch < CLEAN_START_EPOCH:
        return "cooldown"
    return "clean"


def configure_training_phase(epoch):
    phase = training_phase(epoch)
    if phase == "cutmix":
        training_dataset.transform = regularized_train_transform
    elif phase == "cooldown":
        training_dataset.transform = cooldown_train_transform
    else:
        training_dataset.transform = eval_transform
    return phase


def cutmix_loss(images, labels):
    permutation = torch.randperm(images.size(0), device=images.device)
    lam = np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA)
    lam = max(lam, 1.0 - lam)
    height, width = images.shape[-2:]
    cut_ratio = math.sqrt(1.0 - lam)
    cut_width = round(width * cut_ratio)
    cut_height = round(height * cut_ratio)
    center_x = random.randrange(width)
    center_y = random.randrange(height)
    x1 = max(0, center_x - cut_width // 2)
    y1 = max(0, center_y - cut_height // 2)
    x2 = min(width, center_x + cut_width // 2)
    y2 = min(height, center_y + cut_height // 2)

    mixed = images.clone()
    mixed[:, :, y1:y2, x1:x2] = images[permutation, :, y1:y2, x1:x2]
    replaced_area = (x2 - x1) * (y2 - y1)
    adjusted_lam = 1.0 - replaced_area / (height * width)
    with torch.amp.autocast(device_type=device.type, enabled=device.type == "cuda"):
        logits = model(mixed)
        return (
            adjusted_lam * regularized_criterion(logits, labels)
            + (1.0 - adjusted_lam)
            * regularized_criterion(logits, labels[permutation])
        )


def plain_loss(images, labels, criterion):
    with torch.amp.autocast(device_type=device.type, enabled=device.type == "cuda"):
        return criterion(model(images), labels)


def train_one_epoch(phase):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0.0
    images_seen = 0
    cutmix_batches = 0
    epoch_images = len(training_loader.dataset)
    total_steps = len(training_loader)
    progress = tqdm(training_loader, desc=f"Phase: {phase}", leave=False)

    for step, (images, labels) in enumerate(progress, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        use_cutmix = (
            phase == "cutmix"
            and random.random() < CUTMIX_PROBABILITY
        )
        criterion = regularized_criterion if phase == "cutmix" else clean_criterion
        if use_cutmix:
            loss = cutmix_loss(images, labels)
            cutmix_batches += 1
        else:
            loss = plain_loss(images, labels, criterion)

        group_start = ((step - 1) // ACCUMULATION_STEPS) * ACCUMULATION_STEPS
        group_size = min(ACCUMULATION_STEPS, total_steps - group_start)
        scaler.scale(loss / group_size).backward()
        if step % ACCUMULATION_STEPS == 0 or step == total_steps:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size = labels.size(0)
        loss_sum += loss.detach().item() * batch_size
        images_seen += batch_size
        progress.set_postfix(
            images=f"{images_seen:,}/{epoch_images:,}",
            loss=f"{loss_sum / images_seen:.4f}",
            cutmix=cutmix_batches,
        )
    return loss_sum / images_seen, images_seen, epoch_images, cutmix_batches


def should_stop(state):
    return (
        RUN_STAGE == "development"
        and state.epoch >= MIN_DEVELOPMENT_EPOCHS
        and state.stale_epochs >= EARLY_STOP_PATIENCE
    )

In [ ]:
# Pipeline 6/7 - Development selects; full refit follows the locked epoch count.
state = load_checkpoint()

for epoch in range(state.epoch, NUM_EPOCHS):
    if epoch >= CUTMIX_STOP_EPOCH and not state.cooldown_started:
        state.stale_epochs = 0
        state.cooldown_started = True
        print("Plateau reset: CutMix off, entering shrinkage cooldown.")
    if epoch >= CLEAN_START_EPOCH and not state.clean_phase_started:
        state.stale_epochs = 0
        state.clean_phase_started = True
        print("Plateau reset: entering clean refinement.")

    phase = configure_training_phase(epoch)
    epoch_lr = optimizer.param_groups[0]["lr"]
    training_loss, images_seen, epoch_images, cutmix_batches = train_one_epoch(phase)

    if RUN_STAGE == "development":
        correct, total, top1 = evaluate_top1(model, validation_loader)
        if correct > state.best_correct:
            state.best_epoch = epoch + 1
            state.best_correct = correct
            state.best_top1 = top1
            state.best_weights = cpu_state_dict(model)
            state.stale_epochs = 0
        else:
            state.stale_epochs += 1

    state.epoch = epoch + 1
    scheduler.step()
    save_checkpoint(state)

    base_log = (
        f"Epoch {epoch + 1:03d}/{NUM_EPOCHS} | phase: {phase} | "
        f"lr: {epoch_lr:.2e} | images: {images_seen:,}/{epoch_images:,} | "
        f"CutMix batches: {cutmix_batches} | train loss: {training_loss:.4f}"
    )
    if RUN_STAGE == "development":
        print(
            f"{base_log} | validation Top-1: {top1:.3f}% ({correct}/{total}) | "
            f"best: {state.best_top1:.3f}% at epoch {state.best_epoch} | "
            f"plateau: {state.stale_epochs}/{EARLY_STOP_PATIENCE}"
        )
    else:
        print(base_log)

    if should_stop(state):
        print(
            f"Early stop after {state.stale_epochs} epochs "
            "without an exact-count improvement."
        )
        break




if RUN_STAGE == "development":
    if state.best_epoch < 1:
        raise RuntimeError("Development finished without a valid best epoch.")

    result = {
        "format_version": FORMAT_VERSION,
        "seed": SEED,
        "best_epoch": state.best_epoch,
        "validation_correct": state.best_correct,
        "validation_total": len(validation_indices),
        "validation_top1": state.best_top1,
        "config": locked_config(),
    }
    result_path = DEVELOPMENT_RESULT_PATHS[SEED]
    result_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    print("Saved development result:", result_path.resolve())

    completed = {
        seed: path for seed, path in DEVELOPMENT_RESULT_PATHS.items() if path.is_file()
    }
    if len(completed) == len(DEVELOPMENT_SEEDS):
        results = [
            json.loads(DEVELOPMENT_RESULT_PATHS[seed].read_text(encoding="utf-8"))
            for seed in DEVELOPMENT_SEEDS
        ]
        if any(item["config"] != locked_config() for item in results):
            raise RuntimeError("Development results contain different configurations.")
        if [item["seed"] for item in results] != list(DEVELOPMENT_SEEDS):
            raise RuntimeError("Development result seeds do not match the protocol.")

        best_epochs = sorted(item["best_epoch"] for item in results)
        scores = np.asarray([item["validation_top1"] for item in results])
        locked_payload = {
            "format_version": FORMAT_VERSION,
            "source_stage": "three_seed_development",
            "full_refit_epochs": best_epochs[len(best_epochs) // 2],
            "epoch_selection": "median_best_epoch",
            "validation_summary": {
                "mean_top1": float(scores.mean()),
                "std_top1": float(scores.std()),
                "min_top1": float(scores.min()),
                "max_top1": float(scores.max()),
                "runs": results,
            },
            "config": locked_config(),
        }
        temporary_path = LOCKED_CONFIG_PATH.with_suffix(".json.tmp")
        temporary_path.write_text(
            json.dumps(locked_payload, indent=2) + "\n",
            encoding="utf-8",
        )
        temporary_path.replace(LOCKED_CONFIG_PATH)
        print(
            f"Locked 3-seed mean/min: {scores.mean():.3f}%/{scores.min():.3f}% | "
            f"full-refit epochs: {locked_payload['full_refit_epochs']}"
        )
        print("Locked config:", LOCKED_CONFIG_PATH.resolve())
    else:
        remaining = [seed for seed in DEVELOPMENT_SEEDS if seed not in completed]
        print("Remaining development seeds:", remaining)

## Lock development, refit all training data, then test once

In [ ]:
if RUN_STAGE == "development":
    print(
        f"Seed {SEED} selected epoch {state.best_epoch}: "
        f"{state.best_correct}/{len(validation_indices)} ({state.best_top1:.3f}%)."
    )
    completed_seeds = [
        seed for seed, path in DEVELOPMENT_RESULT_PATHS.items() if path.is_file()
    ]
    print("Completed development seeds:", completed_seeds)
    print("Official test has not been evaluated.")
    if LOCKED_CONFIG_PATH.is_file():
        print("Three-seed lock is ready; start full_refit in a fresh runtime.")
    else:
        print("Repeat development in fresh runtimes for the remaining seeds.")
else:
    if state.epoch != NUM_EPOCHS:
        raise RuntimeError(
            f"Full refit is incomplete: {state.epoch}/{NUM_EPOCHS} epochs."
        )
    print(f"Full refit complete: {state.epoch} fixed epochs on all training data.")
    print("Final evaluation is single-view; no ROI or multi-seed ensemble is used.")

In [ ]:
# Pipeline 7/7 - Development never creates a test loader; full refit tests once.
if RUN_STAGE == "development":
    print("Test skipped in development mode.")
elif state.official_test_result is not None:
    print("Official test was already evaluated; cached result:")
    print(state.official_test_result)
else:
    if state.epoch != NUM_EPOCHS:
        raise RuntimeError("Complete full refit before official test evaluation.")
    test_dataset = torchvision.datasets.ImageFolder(
        TEST_ROOT,
        transform=eval_transform,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        **loader_options,
    )
    test_correct, test_total, test_top1 = evaluate_top1(model, test_loader)
    state.official_test_result = {
        "evaluation_mode": "single_view",
        "correct": test_correct,
        "total": test_total,
        "top1": test_top1,
        "target_achieved": test_top1 > TARGET_TOP1,
    }
    save_checkpoint(state)
    print("Evaluation mode: single view")
    print(
        f"Top-1 accuracy: {test_correct}/{test_total} "
        f"({test_top1:.3f}%)"
    )
    print(
        f"Target > {TARGET_TOP1:.1f}%: "
        f"{'achieved' if test_top1 > TARGET_TOP1 else 'not achieved'}."
    )